In [1]:
# Install system downloader (aria2) for fast/robust large-file downloads
!apt-get -qq update
!apt-get -qq install -y aria2

# Install Python helpers
!pip -q install requests tqdm ijson

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.11) ...
/sbin/ldconfig.real: /usr/

In [2]:
# Mount Google Drive so we can write files into MyDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Define Dropbox file links (direct file links: scl/fi)
URLS = {
    "train_with_neg_v0.json": "https://www.dropbox.com/scl/fi/67z8ek28e0tl1x54938qa/train_with_neg_v0.json?rlkey=0sr0hrgj7nmv6jsuscbjzm1ng&st=g7ny25du&dl=0",
    "val_with_neg_v0.json":   "https://www.dropbox.com/scl/fi/cqjb7nj8yp18z64ddabyo/val_with_neg_v0.json?rlkey=38z2s27blcb9olegzju6q1ye0&st=fid7c9ft&dl=0",
}

# Create target folder: MyDrive/final_project/baseline
from pathlib import Path
TARGET_DIR = Path("/content/drive/MyDrive/final_project/baseline")
TARGET_DIR.mkdir(parents=True, exist_ok=True)

print("Target directory:", TARGET_DIR)

Target directory: /content/drive/MyDrive/final_project/baseline


In [4]:
# Download files to local Colab storage first (faster), then move to Google Drive

import subprocess
import shutil
from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse
from pathlib import Path

LOCAL_DIR = Path("/content/tmp_downloads")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def force_dl_1(url: str) -> str:
    # Ensure dl=1 for direct download
    parts = urlparse(url)
    q = dict(parse_qsl(parts.query, keep_blank_values=True))
    q["dl"] = "1"
    new_query = urlencode(q, doseq=True)
    return urlunparse(parts._replace(query=new_query))

def aria2_download(url: str, out_path: Path):
    # Use aria2c for fast/reliable download with multiple connections
    cmd = [
        "aria2c",
        "-x", "16",                      # connections per server
        "-s", "16",                      # number of splits
        "-k", "1M",                      # segment size
        "--file-allocation=none",        # avoid slow pre-allocation
        "--allow-overwrite=true",
        "--retry-wait=5",
        "--max-tries=10",
        "-o", out_path.name,
        "-d", str(out_path.parent),
        url
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

for fname, url in URLS.items():
    direct_url = force_dl_1(url)

    local_path = LOCAL_DIR / fname
    drive_path = TARGET_DIR / fname

    print("=" * 80)
    print("File:", fname)
    print("Direct URL:", direct_url)

    # Download to local path
    aria2_download(direct_url, local_path)

    # Move to Drive (overwrite if exists)
    if drive_path.exists():
        drive_path.unlink()
    shutil.move(str(local_path), str(drive_path))

    print("Saved to Drive:", drive_path)

print("\nAll files downloaded and moved to Google Drive.")

File: train_with_neg_v0.json
Direct URL: https://www.dropbox.com/scl/fi/67z8ek28e0tl1x54938qa/train_with_neg_v0.json?rlkey=0sr0hrgj7nmv6jsuscbjzm1ng&st=g7ny25du&dl=1
Running: aria2c -x 16 -s 16 -k 1M --file-allocation=none --allow-overwrite=true --retry-wait=5 --max-tries=10 -o train_with_neg_v0.json -d /content/tmp_downloads https://www.dropbox.com/scl/fi/67z8ek28e0tl1x54938qa/train_with_neg_v0.json?rlkey=0sr0hrgj7nmv6jsuscbjzm1ng&st=g7ny25du&dl=1
Saved to Drive: /content/drive/MyDrive/final_project/baseline/train_with_neg_v0.json
File: val_with_neg_v0.json
Direct URL: https://www.dropbox.com/scl/fi/cqjb7nj8yp18z64ddabyo/val_with_neg_v0.json?rlkey=38z2s27blcb9olegzju6q1ye0&st=fid7c9ft&dl=1
Running: aria2c -x 16 -s 16 -k 1M --file-allocation=none --allow-overwrite=true --retry-wait=5 --max-tries=10 -o val_with_neg_v0.json -d /content/tmp_downloads https://www.dropbox.com/scl/fi/cqjb7nj8yp18z64ddabyo/val_with_neg_v0.json?rlkey=38z2s27blcb9olegzju6q1ye0&st=fid7c9ft&dl=1
Saved to Drive: /

In [5]:
# Verify that files exist in the Drive folder and print count + sizes

files = sorted(TARGET_DIR.glob("*.json"))
print("Number of .json files in target folder:", len(files))
print("-" * 60)

for p in files:
    size_mb = p.stat().st_size / (1024**2)
    size_gb = p.stat().st_size / (1024**3)
    print(f"{p.name:30s}  {size_mb:10.2f} MB  ({size_gb:.3f} GB)")

Number of .json files in target folder: 2
------------------------------------------------------------
train_with_neg_v0.json             1020.77 MB  (0.997 GB)
val_with_neg_v0.json                 84.07 MB  (0.082 GB)


In [6]:
# Show one sample record from each file without loading the entire file into memory.
# Strategy:
# 1) Try JSON Lines (first non-empty line as JSON)
# 2) If not JSONL, try streaming JSON array with ijson (root: [])
# 3) If not root array, try common "data" array path (root: {"data":[...]})

import json
import ijson

def sample_from_json_file(path: Path, max_chars_preview: int = 1200):
    # First: try JSONL (read first non-empty line)
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(50):  # scan a few lines
            line = f.readline()
            if not line:
                break
            line = line.strip()
            if line:
                try:
                    obj = json.loads(line)
                    return ("jsonl_first_line", obj)
                except Exception:
                    break  # not JSONL or first line is not a JSON object

    # Second: try root array streaming: items under 'item'
    try:
        with open(path, "rb") as f:
            it = ijson.items(f, "item")
            first_item = next(it)
            return ("json_array_first_item", first_item)
    except Exception:
        pass

    # Third: try common path: 'data.item'
    try:
        with open(path, "rb") as f:
            it = ijson.items(f, "data.item")
            first_item = next(it)
            return ("json_data_array_first_item", first_item)
    except Exception:
        pass

    # Fallback: show a raw text preview (first bytes)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        preview = f.read(max_chars_preview)
    return ("raw_preview", preview)

for fname in ["train_with_neg_v0.json", "val_with_neg_v0.json"]:
    p = TARGET_DIR / fname
    mode, sample = sample_from_json_file(p)

    print("=" * 100)
    print("File:", fname)
    print("Sample mode:", mode)
    print("-" * 100)

    # Print sample in a readable way
    if isinstance(sample, (dict, list)):
        print(json.dumps(sample, ensure_ascii=False, indent=2)[:4000])  # limit printed chars
    else:
        print(sample)

File: train_with_neg_v0.json
Sample mode: jsonl_first_line
----------------------------------------------------------------------------------------------------
{
  "question": "Which magazine was started first Arthur's Magazine or First for Women?",
  "answers": [
    "Arthur's Magazine"
  ],
  "type": "comparison",
  "pos_paras": [
    {
      "title": "Arthur's Magazine",
      "text": "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others. In May 1846 it was merged into \"Godey's Lady's Book\"."
    },
    {
      "title": "First for Women",
      "text": "First for Women is a woman's magazine published by Bauer Media Group in the USA. The magazine was started in 1989. It is based in Englewood Cliffs, New Jersey. In 2011 the circulation of the magazine was 1,310,696 copies."
    }
  ],
  "neg_paras": [
    